In [1]:
#from pylibCZIrw import czi as pyczi
#import ipywidgets as widgets
# import json
# from matplotlib import pyplot as plt
# import matplotlib.cm as cm
# import numpy as np
# import os, sys
# from tqdm import tqdm
# from tqdm.contrib import itertools as it
# from matplotlib.patches import Rectangle
# from aicsimageio import AICSImage
# from aicsimageprocessing import resize
# from aicsimageio.writers import OmeTiffWriter
# from aicsimageio.types import PhysicalPixelSizes
import processing_tools as pt

# show the used python env
#print("Using:", sys.executable)

21-Aug-24 21:46:59 - bfio.backends - WARNING  - Java backend is not available. This could be due to a missing dependency (jpype).
c:\Users\amarcosv\AppData\Local\miniforge3\envs\cellpose_mikala_pipeline\lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:

# Folder containing the input data
INPUT_FOLDER = 'D:\\Projects\\Mikala\\images\\raw\\test'
PREPROCESSED_FOLDER = 'D:\\Projects\\Mikala\\images\\downsampled\\test'
OUTPUT_FOLDER = 'D:/Projects/Mikala/output/test'

targetPxSize = [0.25, 0.14, 0.14] #XYZ
channelMapping = '1234' # 1:VASA, 2:membrane, 3:TJ, 4:DAPI

VASA_model_path = 'D:/Projects/Mikala/images/training_bright/single_channel/ch1/xy_seg_model/models/models'
VASA_model_name = 'VASA_xy_pipeline_bright_Mikala'
TJ_model_path =  "D:/Projects/Mikala/images/training_norm/models/models/"
TJ_model_name = 'TJ_xy_pipeline_norm_Mikala'


  

In [3]:

fileList = pt.readFolder(INPUT_FOLDER)

print('Preparing output directory:' )
preprocDatasetFolder = pt.buildOutputFolder(PREPROCESSED_FOLDER, fileList[0].split('_')[0]+'_test')

fullPathFileList = [INPUT_FOLDER  + '\\' + f for f in fileList ]

print('Preprocessing started' )
pt.processFiles(fullPathFileList, preprocDatasetFolder, targetPxSize, channelMapping)


print('Initializing cellpose:')
pt.initializeCellPose()

print('Running cellpose segmentation')
VASA_outputFolder = pt.processVASA(preprocDatasetFolder, OUTPUT_FOLDER, VASA_model_path, VASA_model_name)
TJ_outputFolder = pt.processTJ(preprocDatasetFolder, OUTPUT_FOLDER, TJ_model_path, TJ_model_name)


print('Processing finished')





Preprocessing files from directory: D:\Projects\Mikala\images\raw\test
2 files found:
	08212024_1A_processed_ovary1_63x_dapi_488_568_647.czi
	08212024_1A_processed_ovary2_63x_dapi_488_568_647.czi
Preparing output directory:
	[buildOutputFolder] Creating:D:\Projects\Mikala\images\downsampled\test
Preprocessing started
[processFiles] processing file 1 of 2
	Processing channel 1(DAPI) as channel 1
		Resizing image from [1098, 1098, 148] to [554 554 148]
		Saving resized file as D:\Projects\Mikala\images\downsampled\test\08212024_test\ch1\08212024_1A_processed_ovary1_ch1_downs.tiff
	Processing channel 2(TJ) as channel 2
		Resizing image from [1098, 1098, 148] to [554 554 148]
		Saving resized file as D:\Projects\Mikala\images\downsampled\test\08212024_test\ch2\08212024_1A_processed_ovary1_ch2_downs.tiff
	Processing channel 3(membrane) as channel 3
		Resizing image from [1098, 1098, 148] to [554 554 148]
		Saving resized file as D:\Projects\Mikala\images\downsampled\test\08212024_test\ch3\0

100%|██████████| 147/147 [00:00<00:00, 159.92it/s]


Saving output files to directory:D:/Projects/Mikala/output/test\VASA_ch1\08212024
Processing file 2 of 2
Performing prediction on: D:/Projects/Mikala/output/test\VASA_ch1\08212024\08212024_1A_processed_ovary2_ch1_downs.tiff


100%|██████████| 135/135 [00:00<00:00, 360.77it/s]


Saving output files to directory:D:/Projects/Mikala/output/test\VASA_ch1\08212024
Processing TJ channel:
Cellpose model path = D:/Projects/Mikala/images/training_norm/models/models/TJ_xy_pipeline_norm_Mikala
Processing file 1 of 2
Performing prediction on: D:/Projects/Mikala/output/test\TJ_ch3\08212024\08212024_1A_processed_ovary1_ch3_downs.tiff


100%|██████████| 147/147 [00:00<00:00, 566.05it/s]


Saving output files to directory:D:/Projects/Mikala/output/test\TJ_ch3\08212024
Processing file 2 of 2
Performing prediction on: D:/Projects/Mikala/output/test\TJ_ch3\08212024\08212024_1A_processed_ovary2_ch3_downs.tiff


100%|██████████| 135/135 [00:00<00:00, 286.07it/s]


Saving output files to directory:D:/Projects/Mikala/output/test\TJ_ch3\08212024
Processing finished


In [5]:
VASAstats = pt.processSegmentationMasks(VASA_outputFolder, 'VASA')
TJstats = pt.processSegmentationMasks(TJ_outputFolder, 'TJ')

exportdata = pt.mergeSegmentationResults(TJstats, VASAstats)
pt.exportSegmentationResults(exportdata, OUTPUT_FOLDER, fileList[0].split('_')[0])



KeyError: 'num_pixels'

In [7]:
import os
import pandas as pd  
from skimage import ( feature, io, measure,
                      morphology,  transform)



minSize = 100
label = 'VASA'   
masksFolder = VASA_outputFolder

masks = []
for file in os.listdir(masksFolder):
    # check only text files
    if file.endswith('masks.tif'):
        masks.append(file)


segmentedRegions = pd.DataFrame()
for maskFile in masks:
    f = os.path.join(masksFolder,maskFile)
    mask = io.imread(os.path.join(masksFolder,maskFile))
    props = pd.DataFrame(measure.regionprops_table(mask, properties =['label', 'num_pixels']))
    props['dataset'] = maskFile
    props['date'] = maskFile.split('_')[0]   

    segmentedRegions= pd.concat([segmentedRegions, props],ignore_index=True)

segmentedRegions = segmentedRegions[segmentedRegions['num_pixels'] > minSize ]

regionStats = segmentedRegions.groupby(['dataset','date'],as_index=False, group_keys=False).size()
regionStats['marker']= label   



KeyError: 'num_pixels'